In [ ]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_beus_articulatory_features

In [ ]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
token_df = dataset_df_to_token_df(df)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = VideoAutoencoder(
    input_shape=(10, 56, 112),
    input_channels=2,
    latent_dim=1024,
    use_mlp=True,
    stn_mode="separate",
    grid_sample_cpu_fallback=True,
).to(device)
art_model.load_state_dict(torch.load("../encoder_training/ckpts/beus/art_model_ckpt.pth", map_location=device))
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)
ac_model.load_state_dict(torch.load("../encoder_training/ckpts/beus/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (enc_conv): Sequential(
    (0): Conv1d(30, 512, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(512, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
  )
  (enc_pool): AdaptiveAvgPool1d(output_size=1)
  (fc_enc): Sequential(
    (0): Linear(in_features=128, out_features=4096, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4096, out_features=2048, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2048, out_features=1024, bias=True)
  )
  (fc_dec): Sequential(
    (0): Linear(in_features=1024, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(i

In [4]:
def get_vectors(art_f, ac_f, start, end):
    articulatory_features = get_beus_articulatory_features(
        art_f,
        start,
        end
    )

    if articulatory_features is not None:

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        art_out, art_z = art_model(
            articulatory_features,
            return_latent=True
        )
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = (
            torch.from_numpy(mfccs.T)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs)
    else:
        ac_out, ac_z = None, None

    return {
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [5]:
token_df

,start,end,phone,art_f,ac_f,subject
0,1.83,1.97,k,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
1,2.31,2.39,t,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
2,2.50,2.56,l,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
3,1.75,1.83,z,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
4,2.16,2.17,l,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
...,...,...,...,...,...,...
226483,2.67,2.73,l,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
226484,2.73,2.79,i,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
226485,2.23,2.32,h,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
226486,2.32,2.39,i,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me


In [6]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]

    articulatory_features = get_beus_articulatory_features(art_f, start, end)
    if articulatory_features is None: continue
    articulatory_features = torch.from_numpy(articulatory_features).unsqueeze(0).to(device)
    art_out, art_z = art_model(articulatory_features, return_latent=True)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = torch.from_numpy(mfccs.T).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)

100%|██████████| 226488/226488 [3:30:13<00:00, 17.96it/s]  


In [7]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (226488, 1024) ac_flat: (226488, 1024)


In [8]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

N = len(ac_raw)
k = 5

indices = np.arange(N)
np.random.shuffle(indices)
folds = np.array_split(indices, k)

In [9]:
art_dim = art_raw.shape[1]
ac_dim = ac_raw.shape[1]

In [10]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------
class AXDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# ----------------------------------------------------
# Training loop for one model
# ----------------------------------------------------
def train_model(train_loader, val_loader, x_dim, y_dim):
    model = MDN(x_dim=x_dim, y_dim=y_dim, n_components=5, hidden_dim=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = float('inf')
    patience = 10
    counter = 0

    for epoch in range(1, 2001):  # just large upper bound
        # ----- train -----
        model.train()
        train_loss = 0
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logp = model.log_prob(Xb, Yb)
            loss = -logp.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(Xb)
        train_loss /= len(train_loader.dataset)

        # ----- validation -----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                loss = -model.log_prob(Xb, Yb).mean()
                val_loss += loss.item() * len(Xb)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
            print("   ✓ New best model")
        else:
            counter += 1
            print(f"   No improvement ({counter} epochs)")

        if counter >= patience:
            print(f"Early stopping. Best val={best_val:.4f}")
            break

    model.load_state_dict(best_state)
    return model

In [11]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]


    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_ac = StandardScaler()
    sc_art = StandardScaler()

    sc_ac.fit(ac_raw[train_inner])
    sc_art.fit(art_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    ac_norm = sc_ac.transform(ac_raw)
    art_norm = sc_art.transform(art_raw)

    # Dataset for p(ART | AC)
    dataset = AXDataset(ac_norm, art_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=ac_raw.shape[1],
        y_dim=art_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            ac_fold,
            art_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    ART coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_art.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_art_given_ac_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_art_given_ac_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=2.2811, val=-120.5419
   ✓ New best model
Epoch 2: train=-268.5931, val=-334.8279
   ✓ New best model
Epoch 3: train=-349.1352, val=-358.1017
   ✓ New best model
Epoch 4: train=-358.5061, val=-361.1031
   ✓ New best model
Epoch 5: train=-359.1564, val=-362.6731
   ✓ New best model
Epoch 6: train=-363.3314, val=-363.6431
   ✓ New best model
Epoch 7: train=-362.7575, val=-362.3742
   No improvement (1 epochs)
Epoch 8: train=-354.1944, val=-334.9473
   No improvement (2 epochs)
Epoch 9: train=-365.3902, val=-365.6562
   ✓ New best model
Epoch 10: train=-369.2968, val=-365.1111
   No improvement (1 epochs)
Epoch 11: train=-370.8761, val=-364.5555
   No improvement (2 epochs)
Epoch 12: train=-371.0587, val=-365.9617
   ✓ New best model
Epoch 13: train=-371.1534, val=-365.3861
   No improvement (1 epochs)
Epoch 14: train=-371.2166, val=-364.9513
   No improvement (2 epochs)
Epoch 15: train=-371.5120, val=-363.8814
   No improvement (3 epochs)
Epoch 16: train=-370.93

In [12]:
import pickle
with open("entropies/beus_art_given_ac.pickle", "wb") as f:
    pickle.dump(H_art_given_ac_mdn, f)

In [13]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]

    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_art = StandardScaler()
    sc_ac = StandardScaler()

    sc_art.fit(art_raw[train_inner])
    sc_ac.fit(ac_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    art_norm = sc_art.transform(art_raw)
    ac_norm = sc_ac.transform(ac_raw)

    # Dataset for p(AC | ART)
    dataset = AXDataset(art_norm, ac_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=art_raw.shape[1],
        y_dim=ac_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            art_fold,
            ac_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    AC coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_ac.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_ac_given_art_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_ac_given_art_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=-258.2989, val=-539.1684
   ✓ New best model
Epoch 2: train=-531.3842, val=-581.1269
   ✓ New best model
Epoch 3: train=-574.3545, val=-598.7550
   ✓ New best model
Epoch 4: train=-573.3408, val=-602.7052
   ✓ New best model
Epoch 5: train=-586.7110, val=-602.1161
   No improvement (1 epochs)
Epoch 6: train=-587.4741, val=-589.8920
   No improvement (2 epochs)
Epoch 7: train=-588.4969, val=-603.1687
   ✓ New best model
Epoch 8: train=-590.9688, val=-600.7872
   No improvement (1 epochs)
Epoch 9: train=-590.7515, val=-602.6263
   No improvement (2 epochs)
Epoch 10: train=-591.7424, val=-602.4736
   No improvement (3 epochs)
Epoch 11: train=-590.2265, val=-603.4014
   ✓ New best model
Epoch 12: train=-594.0888, val=-593.5405
   No improvement (1 epochs)
Epoch 13: train=-593.3491, val=-605.6882
   ✓ New best model
Epoch 14: train=-594.5794, val=-603.9735
   No improvement (1 epochs)
Epoch 15: train=-593.7982, val=-601.2344
   No improvement (2 epochs)
Epoch 16: t

In [14]:
with open("entropies/beus_ac_given_art.pickle", "wb") as f:
    pickle.dump(H_ac_given_art_mdn, f)